# XOR Classifier with Manual Backpropagation

## Problem Statement
Train an XOR classifier from scratch using manual backpropagation.
The XOR (Exclusive OR) problem is a classic example that linear classifiers cannot solve. It requires a non-linear activation function and a hidden layer.

**Inputs**: `[[0,0], [0,1], [1,0], [1,1]]`
**Labels**: `[0, 1, 1, 0]`

## Steps to Solve the Problem
1.  **Initialization**: Setup the Neural Network architecture (2-input, 2-hidden, 1-output). Initialize weights randomly and biases to zero.
2.  **Forward Propagation**: 
    - Compute Hidden Layer Linear Step (Z1).
    - Apply Activation (ReLU) -> A1.
    - Compute Output Layer Linear Step (Z2).
    - Apply Activation (Sigmoid) -> A2 (Prediction).
3.  **Loss Calculation**: Compare Prediction (A2) with True Label (Y) using Binary Cross-Entropy.
4.  **Backward Propagation**:
    - Calculate gradients for Output Layer (dZ2, dW2, db2).
    - Propagate error to Hidden Layer (dA1).
    - Calculate gradients for Hidden Layer (dZ1, dW1, db1).
5.  **Optimization (Gradient Descent)**: Update weights and biases using the gradients and learning rate.
6.  **Gradient Check**: Verify the manual backprop implementation using Finite Difference method.

## Expected Output
- Loss should decrease over 5000 iterations (target < 0.02).
- Predictions should match the XOR truth table with high confidence (> 0.95).
- Gradient check error should be minimal (< 1e-3).

### 1. Import Libraries

**2.1 Definition**: Import the NumPy library.
**2.2 Why**: NumPy is the fundamental package for scientific computing in Python. It provides support for arrays and matrices.
**2.3 When**: Always used when performing linear algebra, vectorization, or mathematical operations on data.
**2.4 Where**: At the very beginning of the script.
**2.5 How to use**: `import numpy as np` allows accessing functions via `np.function()`.
**2.6 How it works**: Loads the C-optimized math library into memory.
**2.7 Output**: Module object 'np'.

**2.1 Definition**: Set the random seed.
**2.2 Why**: To ensure reproducibility. Random numbers will be the same every time code is run.
**2.3 When**: Before generating any random numbers.
**2.4 Where**: Global scope or start of execution.
**2.5 How to use**: `np.random.seed(integer_value)`.
**2.6 How it works**: Initializes the pseudo-random number generator state with the given integer.
**2.7 Output**: None.

In [ ]:
import numpy as np
np.random.seed(7)

### 2. Activation Function: Sigmoid

**2.1 Definition**: Sigmoid Activation Function using standard formula 1 / (1 + e^-z).
**2.2 Why**: Maps input to a probability range (0 to 1). Crucial for binary classification.
**2.3 When**: Used as the activation output of the final layer.
**2.4 Where**: Last step of forward pass.
**2.5 How to use**: `s = 1 / (1 + np.exp(-z))`.
**2.6 How it works**: Exponentiates the negative input, adds 1, and inverts. Large pos -> 1, Large neg -> 0.
**2.7 Output**: Float value between 0 and 1.

**Arguments**:
*   **3.1 Argument `z`**: Scalar or NumPy array.
*   **3.2 Why**: The input value(s) to be squashed.
*   **3.3 When**: During Forward Propagation (Output Layer).
*   **3.4 Where**: Passed from the linear combination of weights and inputs.
*   **3.5 How to use**: `a = sigmoid(z)`.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = 1 / (1 + np.exp(-z))
    return s * (1 - s)

### 3. Activation Function: ReLU

**2.1 Definition**: Rectified Linear Unit (ReLU).
**2.2 Why**: Introduces non-linearity without the vanishing gradient problem of sigmoid for deep layers.
**2.3 When**: Hidden layers activation.
**2.4 Where**: Between Linear Step 1 and Linear Step 2.
**2.5 How to use**: `np.maximum(0, z)`.
**2.6 How it works**: Returns z if z > 0, else 0.
**2.7 Output**: Array with negative values replaced by 0.

**Arguments**:
*   **3.1 Argument `z`**: Input array.
*   **3.2 Why**: The linear output from the previous layer.
*   **3.3 When**: During Forward Propagation (Hidden Layer).

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    dZ = np.array(z, copy=True)
    dZ[z <= 0] = 0
    dZ[z > 0] = 1
    return dZ

### 4. Dataset Creation

**2.1 Definition**: Define Input Features (X) and Labels (Y).
**2.2 Why**: The XOR problem consists of 4 possible inputs. Supervised learning requires targets to calculate error.
**2.3 When**: Before training starts.
**2.4 Where**: Global scope.
**2.5 How to use**: Numpy array of shape (4, 2) for X, (4, 1) for Y.
**2.6 How it works**: Creates a matrix in memory.
**2.7 Output**: `X` matrix and `Y` column vector.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
Y = np.array([[0], [1], [1], [0]])

### 5. Parameter Initialization

**2.1 Definition**: Initialize Weights and Biases for the Neural Network.
**2.2 Why**: Weights need to be random to break symmetry (learn different features). Biases start at zero.
**2.3 When**: Start of training.
**2.4 Where**: `initialize_parameters` function.
**2.5 How to use**: `parameters = initialize_parameters()`.
**2.6 How it works**: Uses `np.random.randn` scaled by 0.5.
**2.7 Output**: Dictionary containing `W1`, `b1`, `W2`, `b2`.

In [ ]:
def initialize_parameters():
    W1 = np.random.randn(2, 2) * 0.5
    b1 = np.zeros((1, 2))
    W2 = np.random.randn(2, 1) * 0.5
    b2 = np.zeros((1, 1))
    
    parameters = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}
    return parameters

### 6. Forward Propagation

**2.1 Definition**: Forward pass mechanism.
**2.2 Why**: To calculate the prediction (A2) given inputs (X).
**2.3 When**: Every training iteration and during inference.
**2.4 Where**: `forward_propagation` function.
**2.5 How to use**: `A2, cache = forward_propagation(X, params)`.
**2.6 How it works**: Computes Dot Product -> Add Bias -> Activation -> Repeat.
**2.7 Output**: Prediction `A2` and `cache` (Z1, A1, Z2, A2) for backprop.

**Arguments**:
*   **3.1 Argument `X`**: Input data.
*   **3.2 Argument `parameters`**: Dictionary of weights/biases.

In [ ]:
def forward_propagation(X, parameters):
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)
    
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

### 7. Loss Calculation

**2.1 Definition**: Binary Cross-Entropy Loss (Log Loss).
**2.2 Why**: Standard loss for binary classification. Penalizes wrong confident predictions accurately.
**2.3 When**: After forward pass.
**2.4 Where**: `compute_loss` function.
**2.5 How to use**: `loss = compute_loss(A2, Y)`.
**2.6 How it works**: Sum of `Y*log(P)` terms.
**2.7 Output**: Scalar float representing error.

**Arguments**:
*   **3.1 Argument `A2`**: Predicted probabilities.
*   **3.2 Argument `Y`**: True labels.

In [ ]:
def compute_loss(A2, Y):
    m = Y.shape[0]
    logprobs = np.multiply(Y, np.log(A2)) + np.multiply((1 - Y), np.log(1 - A2))
    loss = -1/m * np.sum(logprobs)
    return np.squeeze(loss)

### 8. Backward Propagation

**2.1 Definition**: Backward pass logic (The Learning Step).
**2.2 Why**: To calculate gradients (derivatives) of the Loss with respect to Weights to minimize error.
**2.3 When**: After Loss calculation.
**2.4 Where**: `backward_propagation` function.
**2.5 How to use**: `grads = backward_propagation(params, cache, X, Y)`.
**2.6 How it works**: Uses Chain Rule from Output layer back to Input layer.
**2.7 Output**: Dictionary `grads` containing `dW1`, `db1`, `dW2`, `db2`.

**Arguments**:
*   **3.1 Argument `parameters`**: Current weights.
*   **3.2 Argument `cache`**: Values from forward pass needed for derivatives.
*   **3.3 Argument `X`, `Y`**: Data and Labels.

In [ ]:
def backward_propagation(parameters, cache, X, Y):
    m = X.shape[0]
    W1, W2 = parameters["W1"], parameters["W2"]
    A1, A2 = cache["A1"], cache["A2"]
    Z1 = cache["Z1"]
    
    dZ2 = A2 - Y
    dW2 = (1/m) * np.dot(A1.T, dZ2)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)
    
    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = (1/m) * np.dot(X.T, dZ1)
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)
    
    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

### 9. Parameter Update

**2.1 Definition**: Gradient Descent Update Rule.
**2.2 Why**: To actually modify the model parameters to learn.
**2.3 When**: After calculating gradients.
**2.4 Where**: `update_parameters` function.
**2.5 How to use**: `params = update_parameters(params, grads, lr)`.
**2.6 How it works**: `W_new = W_old - alpha * Gradient`.
**2.7 Output**: Updated parameter dictionary.

**Arguments**:
*   **3.1 Argument `grads`**: Calculated gradients.
*   **3.2 Argument `learning_rate`**: Step size (default 0.1).

In [ ]:
def update_parameters(parameters, grads, learning_rate=0.1):
    parameters["W1"] = parameters["W1"] - learning_rate * grads["dW1"]
    parameters["b1"] = parameters["b1"] - learning_rate * grads["db1"]
    parameters["W2"] = parameters["W2"] - learning_rate * grads["dW2"]
    parameters["b2"] = parameters["b2"] - learning_rate * grads["db2"]
    return parameters

### 10. Gradient Check

**2.1 Definition**: Verification of Backpropagation implementation.
**2.2 Why**: To ensure the mathematical derivation and code for backprop is correct.
**2.3 When**: Before full training.
**2.4 Where**: `gradient_check` function.
**2.5 How to use**: `gradient_check(params, X, Y)`.
**2.6 How it works**: Compares Analytical Gradient (backprop) vs Numerical Gradient (Finite Difference).
**2.7 Output**: Difference value (should be < 1e-3).

In [ ]:
def gradient_check(parameters, X, Y, epsilon=1e-4):
    print("\n--- Running Gradient Check ---")
    A2, cache = forward_propagation(X, parameters)
    grads = backward_propagation(parameters, cache, X, Y)
    grad_analytical = grads["dW1"][0, 0]
    
    W1_original = parameters["W1"][0, 0]
    
    parameters["W1"][0, 0] = W1_original + epsilon
    A2_plus, _ = forward_propagation(X, parameters)
    loss_plus = compute_loss(A2_plus, Y)
    
    parameters["W1"][0, 0] = W1_original - epsilon
    A2_minus, _ = forward_propagation(X, parameters)
    loss_minus = compute_loss(A2_minus, Y)
    
    parameters["W1"][0, 0] = W1_original
    grad_numerical = (loss_plus - loss_minus) / (2 * epsilon)
    
    diff = abs(grad_analytical - grad_numerical)
    
    print(f"Analytical Gradient: {grad_analytical:.8f}")
    print(f"Numerical Gradient:  {grad_numerical:.8f}")
    print(f"Difference: {diff:.8f}")
    
    if diff < 1e-3: print(">> Gradient Check PASSED!")
    else: print(">> Gradient Check FAILED!")
    return diff

### 11. Main Execution

**2.1 Definition**: The Training Loop.
**2.2 Why**: Iteratively optimizes the model.
**2.3 When**: Main script execution.
**2.4 Where**: Loop over `iterations`.
**2.5 How to use**: Run the cell.
**2.6 How it works**: Forward -> Loss -> Backward -> Update.
**2.7 Output**: Printed loss logs and final predictions.

In [ ]:
parameters = initialize_parameters()
gradient_check(parameters, X, Y, epsilon=1e-4)

print("\n--- Starting Training ---")
iterations = 5000

for i in range(iterations):
    A2, cache = forward_propagation(X, parameters)
    cost = compute_loss(A2, Y)
    grads = backward_propagation(parameters, cache, X, Y)
    parameters = update_parameters(parameters, grads, learning_rate=0.1)
    
    if i % 500 == 0:
        print(f"Iteration {i}: Loss = {cost:.5f}")

print("\n--- Final Results ---")
A2_final, _ = forward_propagation(X, parameters)
print("Predictions vs Labels:")
print(np.c_[X, Y, np.round(A2_final), A2_final])